# Gini Impurity: Calculate Gini for All Hockey Game Splits

**Dataset:** Hockey Game (14 records)  
**Target column:** `PlayHockey` (`Yes` / `No`)

This notebook evaluates every unique binary split for each input feature and calculates parent Gini, child Gini, weighted child Gini, and Gini reduction.


In [ ]:
import pandas as pd
from itertools import combinations

# Keep hockey_game_dataset.csv in the same folder as this notebook.
DATASET_FILE = "hockey_game_dataset.csv"
df = pd.read_csv(DATASET_FILE)

TARGET = "PlayHockey"
FEATURES = [column for column in df.columns if column != TARGET]

print("Dataset shape:", df.shape)
display(df.head())
print("\nTarget class counts:")
print(df[TARGET].value_counts())

## 1. Gini impurity formula

Gini impurity = 1 - sum of squared class probabilities.

Weighted child Gini is the weighted average of the left and right child impurities. Gini reduction is parent Gini minus weighted child Gini. A larger reduction indicates a better split.


In [ ]:
def gini_impurity(labels):
    """Calculate Gini impurity for a collection of class labels."""
    if len(labels) == 0:
        return 0.0
    probabilities = labels.value_counts(normalize=True)
    return float(1 - (probabilities ** 2).sum())

parent_gini = gini_impurity(df[TARGET])
print("Number of records:", len(df))
print("Parent class counts:")
print(df[TARGET].value_counts())
print(f"\nParent Gini impurity: {parent_gini:.4f}")

## 2. Generate and calculate every unique binary split

For each feature, all non-empty category groupings are considered. Reversed duplicates are counted only once.


In [ ]:
def unique_binary_splits(categories):
    """Return each unique non-empty binary partition exactly once."""
    categories = sorted(list(categories))
    if len(categories) < 2:
        return []
    first = categories[0]
    remaining = categories[1:]
    splits = []
    for size in range(0, len(remaining) + 1):
        for subset in combinations(remaining, size):
            left_categories = {first, *subset}
            right_categories = set(categories) - left_categories
            if right_categories:
                splits.append((left_categories, right_categories))
    return splits

results = []
for feature in FEATURES:
    categories = df[feature].dropna().unique()
    for left_categories, right_categories in unique_binary_splits(categories):
        left_mask = df[feature].isin(left_categories)
        left_df = df[left_mask]
        right_df = df[~left_mask]
        left_gini = gini_impurity(left_df[TARGET])
        right_gini = gini_impurity(right_df[TARGET])
        n_left = len(left_df)
        n_right = len(right_df)
        total = n_left + n_right
        weighted_child_gini = (n_left / total) * left_gini + (n_right / total) * right_gini
        gini_reduction = parent_gini - weighted_child_gini
        results.append({
            "Feature": feature,
            "Left split values": ", ".join(sorted(map(str, left_categories))),
            "Right split values": ", ".join(sorted(map(str, right_categories))),
            "Left records": n_left,
            "Right records": n_right,
            "Parent Gini": parent_gini,
            "Left child Gini": left_gini,
            "Right child Gini": right_gini,
            "Weighted child Gini": weighted_child_gini,
            "Gini reduction": gini_reduction
        })

results_df = pd.DataFrame(results)
numeric_columns = ["Parent Gini", "Left child Gini", "Right child Gini", "Weighted child Gini", "Gini reduction"]
display(results_df.style.format({column: "{:.4f}" for column in numeric_columns}))

## 3. Best split

The best split is the one with the highest Gini reduction, equivalent to the lowest weighted child Gini impurity.


In [ ]:
if not results_df.empty:
    best_split = results_df.loc[results_df["Gini reduction"].idxmax()]
    print("Best split based on Gini reduction")
    print("----------------------------------")
    for column in results_df.columns:
        value = best_split[column]
        if isinstance(value, float):
            print(f"{column}: {value:.4f}")
        else:
            print(f"{column}: {value}")
else:
    print("No valid binary splits were found.")

## 4. Save the complete results

This creates a CSV file containing all calculated hockey feature splits in the notebook's working folder.


In [ ]:
RESULTS_FILE = "susmitha_hockey_gini_results.csv"
results_df.to_csv(RESULTS_FILE, index=False)
print("Saved:", RESULTS_FILE)
print("Total unique binary splits evaluated:", len(results_df))

## 5. Conclusion

- **Parent Gini** measures impurity before splitting.
- **Left child Gini** and **right child Gini** measure impurity in each resulting group.
- **Weighted child Gini** combines both child impurities according to their sizes.
- **Gini reduction** measures how much impurity is removed by a split.
- The split with the highest Gini reduction is preferred.

**Note:** Keep `hockey_game_dataset.csv` in the same folder as this notebook. The target is `PlayHockey`, and the input features are `Venue`, `Temperature`, `IceCondition`, and `TeamForm`.
